In [86]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import TargetEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [81]:
# Separate Values
oneHotColumns = ['Marital status', 'Application mode', 'Daytime/evening attendance', 'Previous qualification'
, 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'International']

numericalColumns = ['Application order', 'Age at enrollment', 'Curricular units 1st sem (credited)'
, 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)'
, 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)'
, 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)'
, 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)', 'Unemployment rate', 'Inflation rate', 'GDP']

targetCols = ['Course', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation"
, "Father's occupation"]

In [98]:
# Model
le = LabelEncoder()

x_train = train.drop(columns=['SampleID', 'Target'])
y_train = le.fit_transform(train['Target'])
x_test = test.drop(columns=['SampleID'])

preprocessor = ColumnTransformer(
    transformers=[
        ('oneHot', OneHotEncoder(drop='first',handle_unknown='ignore'), oneHotColumns),
        ('scale', StandardScaler(), numericalColumns),
        ('target', TargetEncoder(), targetCols)
    ]
)

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', XGBClassifier(random_state=42))
])

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5],
    'model__learning_rate': [0.05, 0.1]
}

gridSearch = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
)

gridSearch.fit(x_train, y_train)
model = gridSearch.best_estimator_
predictions = model.predict(x_test)
predictions = le.inverse_transform(predictions)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unkn

In [100]:
# Dataframe
rows = []
for id,pred in zip(test['SampleID'], predictions):
  rows.append({
    "SampleID": id,
    "Target": pred
  })
subDf = pd.DataFrame(rows)
subDf.to_csv('submission.csv', index=False)